In [1]:
import os

# Define the cache directory
cache_dir = os.path.expanduser("/home/jovyan/buckets")

# Set the environment variables
os.environ["VLLM_CACHE_ROOT"] = cache_dir
os.environ["HF_HUB_CACHE"] = cache_dir
os.environ["HF_HOME"] = cache_dir

In [3]:
import os
import getpass
from huggingface_hub import login

# Prompt for the token secretly
hf_token = getpass.getpass("Enter your HF_TOKEN: ")

# Set it as an environment variable
os.environ["HF_TOKEN"] = hf_token

login(token=hf_token)

Enter your HF_TOKEN:  ········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [15]:
"""
Pre-processing TimeML-style SML XML files
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
• Crawls 1.5/**.xml
• Extracts tokens, mentions, spans and relations
• Builds a HuggingFace Dataset and uploads it
"""

from __future__ import annotations
import os, glob, json
import xml.etree.ElementTree as ET
from typing import List, Dict
from datasets import Dataset

# ---------- low-level helpers -------------------------------------------------
def _sorted_by_int_attr(elems, attr):
    """Return elements sorted by the integer value of `attr`."""
    return sorted(elems, key=lambda el: int(el.get(attr, "0")))

def _parse_single_xml(path: str, exclude: str = []) -> Dict:
    """Parse one SML file and return a single row ready for HF Datasets."""
    tree = ET.parse(path)
    root = tree.getroot()

    doc_id = root.get("doc_name") or os.path.basename(path)
    if os.path.basename(path) in exclude or doc_id in exclude:
        # Skip anything whose basename *or* doc_name is black-listed
        print(f"🚫  Skipping excluded file: {path}")
        return None
    
    # 1. tokens ----------------------------------------------------------------
    tokens_el = _sorted_by_int_attr(root.findall(".//token"), "t_id")
    tokens: List[str] = [tok.text or "" for tok in tokens_el]

    # 2. mentions & spans ------------------------------------------------------
    mentions, spans = [], []
    markables = root.find("Markables") or ET.Element("dummy")         # safety
    for m in markables:
        m_id = m.get("m_id")
        if not m_id:           # skip malformed markables
            continue
        anchors = _sorted_by_int_attr(m.findall("token_anchor"), "t_id")
        t_ids = [a.get("t_id") for a in anchors]
        if len(t_ids)>0:
            mentions.append(m_id)
            spans.append(t_ids)
            t_idx = [int(t) for t in t_ids]
            for i in range(min(t_idx), max(t_idx)+1):
                if i not in t_idx:
                    print("Not a span", path, m_id, t_ids, i)

    # 3. relations -------------------------------------------------------------
    relations: Dict[str, List[List[str]]] = {}
    relations_el = root.find("Relations") # or ET.Element("dummy")
    for rel in relations_el:
        rel_type = rel.get("relType", "UNKNOWN")
        if rel_type == "": # Debug
            print("This is why there is a blank rel type", path) 
        if rel_type == "UNKNOWN": # Debug
            print("This is why there is an UNKNOWN rel type", path) 
        src_id  = rel.findtext("source/@m_id") if rel.find("source") is None else rel.find("source").get("m_id")
        tgt_id  = rel.findtext("target/@m_id") if rel.find("target") is None else rel.find("target").get("m_id")
        if src_id and tgt_id:
            if src_id in mentions and tgt_id in mentions:
                relations.setdefault(rel_type, []).append([src_id, tgt_id])
        # if src_id not in mentions or tgt_id not in mentions: # Debug
        #     print("Missing mentions", path, rel, src_id, tgt_id) 

    # 4. meta ------------------------------------------------------------------
    doc_id = root.get("doc_name") or os.path.basename(path)

    return {
        "id":        doc_id,
        "tokens":    tokens,
        "mentions":  mentions,
        "spans":     spans,
        "relations": relations,
    }

# ---------- dataset builder ---------------------------------------------------
def build_dataset(root_dir: str = "1.5", exclude: str = []) -> Dataset:
    """Walk `root_dir/**.xml`, parse files, and return a HuggingFace Dataset."""
    xml_files = glob.glob(os.path.join(root_dir, "**", "*.xml"), recursive=True)
    if not xml_files:
        raise FileNotFoundError(f"No XML files found under {root_dir}/**.xml")

    rows = [_parse_single_xml(p, exclude) for p in xml_files]
    rows.remove(None)
    
    return Dataset.from_list(rows)

# ---------- push to HuggingFace Hub ------------------------------------------
def push_to_hub(
    dataset: Dataset,
    repo_id: str,
    token: str | None = None,
    private: bool = False,
) -> None:
    """Push the dataset to the HF Hub; `token` can come from the env var HF_TOKEN."""
    dataset.push_to_hub(
        repo_id,
        token=token or os.getenv("HF_TOKEN"),
        private=private,
    )
    print(f"✅ Dataset pushed: https://huggingface.co/datasets/{repo_id}")

In [ ]:
# 1) Build the dataset
ds = build_dataset("data/EventStoryLine/annotated_data/v1.5", exclude=["1_10ecbplus.xml"])

# 2) (Optional) check, split, or shuffle here
ds = ds.shuffle(seed=42)
ds = ds.train_test_split(test_size=0.1)

# 3) Push to Hub – adjust `repo_id` and privacy as you wish
push_to_hub(ds, repo_id="Nofing/EventStoryLine-1.5-span", private=False)

In [21]:
from collections import Counter
from datasets import load_dataset   # only needed if you want to load from Hub

# ------------------------------------------------------------------
# OPTION 1 – you already have `data` as a Python list of dicts
#            (the rows returned by _parse_single_xml).
# data = [...]  # ← list produced by your pipeline

# OPTION 2 – you pushed the dataset to the Hub and want to pull it back
# data = load_dataset("your-username/sml_1_5", split="train")  # or "train+test"

# Pick whichever option applies:
data = load_dataset("Nofing/EventStoryLine-1.5-span", split="train")  # 🔄 adjust if needed
# data = [...]  # ← uncomment if you already have the list in memory
# ------------------------------------------------------------------

support = Counter()

for doc in data:
    # doc["relations"] is a dict: {relType: [[src, tgt], ...], ...}
    for rel_type, links in doc["relations"].items():
        support[rel_type] += len(links) if type(links)==list else 0

# Pretty-print, biggest first
for rel_type, count in support.most_common():
    print(f"{rel_type:<15} : {count}")

AFTER           : 2691
CONTAINS        : 2565
PRECONDITION    : 1160
FALLING_ACTION  : 1114
OVERLAP         : 536
BEFORE          : 277
SIMULTANEOUS    : 80
BEGUN_ON        : 59
ENDED_ON        : 27
UNKNOWN         : 22
ENDS_ON         : 7
BEGINS_ON       : 5


In [16]:
#!/usr/bin/env python
"""
preprocess_ann.py  ─────────────────────────────────────────────────────────────
Parse *.ann.tsvx event-relation files organised as
   <root>/<language>/{train,test,dev}/*.ann.tsvx

For every document we return
    • id         : file name without extension
    • lang       : language code (folder name)
    • split      : "train" | "test" | "dev"
    • tokens     : list[str]
    • events     : list[event_id]
    • spans      : list[list[int]]         (token indices)
    • relations  : dict[str, list[[src, tgt]]]

CLI usage
---------
python preprocess_ann.py \
    --root_dir data/ann_dataset \
    --repo_id  my-user/aviation-events \
    --exclude  aviation_accidents-week4-nhung-7946493_chunk_12.ann.tsvx \
    --shuffle 42 \
    --private
"""

from __future__ import annotations
import os, glob, argparse
from collections import defaultdict
from typing import Dict, List, Set
import re

import datasets
from datasets import Dataset, DatasetDict

# ──────────────────────────────────────────────────────────────────────────────
def _parse_ann_file(path: str) -> Dict:
    """Return one row (dict) extracted from a single *.ann.tsvx file."""
    with open(path, encoding="utf-8") as fh:
        lines = [ln.rstrip("\n") for ln in fh]

    if not lines or not lines[0].startswith("Text\t"):
        raise ValueError(f"Malformed file (no Text line): {path}")

    # ── 1. tokenise text ────────────────────────────────────────────────────
    raw_text = lines[0].split("\t", 1)[1]

    # Use regex to grab every non-whitespace span:
    token_matches = list(re.finditer(r"\S+", raw_text))
    tokens: List[str]  = [m.group(0) for m in token_matches]
    starts: List[int]  = [m.start()   for m in token_matches]   # char → token idx

    # Helper: char-offset  ➜  token-index
    def char_to_tok(char_idx: int) -> int:
        # Linear scan is fine (docs are short), but bisect
        # would be O(log n) if you prefer.
        for i, s in enumerate(starts):
            if s <= char_idx < s + len(tokens[i]):
                return i
        raise ValueError(f"Offset {char_idx} outside text in {path}")

    # ── 2. events & spans ───────────────────────────────────────────────────
    events, spans = [], []
    for ln in lines[1:]:
        if not ln.startswith("Event\t"):
            continue
        # Format:  Event  T0  surviving  EVENT  11
        _, e_id, surface, _label, char_off = ln.split("\t")[:5]
        off = int(char_off)

        tok_idx = char_to_tok(off)

        # If the surface form contains spaces, the mention is multi-token.
        length = len(surface.split())
        spans.append([tok_idx, tok_idx + length])
        events.append(e_id)

    # 3. relations -------------------------------------------------------------
    relations: Dict[str, List[List[str]]] = defaultdict(list)
    for ln in lines[1:]:
        if not ln.startswith("Relation\t"):
            continue
        _, src, tgt, rel_type, *_ = ln.split("\t")
        relations[rel_type].append([src, tgt])

    return {
        "id":        os.path.splitext(os.path.basename(path))[0],
        "tokens":    tokens,
        "events":    events,
        "spans":     spans,
        "relations": dict(relations),
    }

# ──────────────────────────────────────────────────────────────────────────────
def build_dataset_dict(root_dir: str, exclude: Set[str]) -> DatasetDict:
    """Walk every <language>/<split> directory and build a DatasetDict."""
    split_rows = defaultdict(list)          # split → list[dict]

    for lang_dir in glob.glob(os.path.join(root_dir, "*")):
        if not os.path.isdir(lang_dir):
            continue
        lang = os.path.basename(lang_dir)

        for split in ("train", "test", "dev"):
            pattern = os.path.join(lang_dir, split, "*.ann.tsvx")
            for path in glob.glob(pattern):
                if os.path.basename(path) in exclude:
                    print(f"🚫  Skipping excluded file: {path}")
                    continue
                row = _parse_ann_file(path)
                row["lang"]  = lang
                row["split"] = split
                split_rows[split].append(row)

    if not split_rows:
        raise RuntimeError(f"No documents found under {root_dir}")

    # Convert to HuggingFace datasets
    return DatasetDict({sp: Dataset.from_list(rows) for sp, rows in split_rows.items()})

# ──────────────────────────────────────────────────────────────────────────────
def push_to_hub(dd: DatasetDict, repo_id: str, private: bool, token: str | None):
    dd.push_to_hub(repo_id,
                   token=token or os.getenv("HF_TOKEN"),
                   private=private)
    print(f"✅  Dataset pushed to https://huggingface.co/datasets/{repo_id}")

In [44]:
exclude_set = []

ds_dict = build_dataset_dict("data/MECI/meci-v0.1-public", exclude_set)

# Optional shuffle for reproducible training

for sp in ds_dict:
    ds_dict[sp] = ds_dict[sp].shuffle(seed=42)

push_to_hub(ds_dict,
            repo_id="Nofing/MECI-v0.1-public-span",
            private=False,
            token=None)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅  Dataset pushed to https://huggingface.co/datasets/Nofing/MECI-v0.1-public-span


In [43]:
ds_dict['dev'][1]#['tokens'][103:108]

{'id': 'aviation_accidents-week4-phuong-1154033_chunk_11.ann',
 'tokens': ['The',
  'accident',
  'was',
  'investigated',
  'by',
  'the',
  'Indonesian',
  'NTSC',
  ',',
  'which',
  'was',
  'assisted',
  'by',
  'expert',
  'groups',
  'from',
  'the',
  'US',
  ',',
  'Singapore',
  ',',
  'and',
  'Australia',
  '.',
  'Around',
  '73',
  '%',
  'of',
  'the',
  'wreckage',
  '(',
  'by',
  'weight',
  ')',
  'was',
  'recovered',
  ',',
  'partially',
  'reconstructed',
  ',',
  'and',
  'examined',
  '.',
  'Both',
  'of',
  'the',
  'aircraft',
  'recorders',
  ',',
  'the',
  'CVR',
  'and',
  'the',
  'FDR',
  ',',
  'were',
  'retrieved',
  'from',
  'the',
  'river',
  'and',
  'their',
  'data',
  'were',
  'extracted',
  'and',
  'analyzed',
  '.',
  'The',
  'investigators',
  'tested',
  '20',
  'different',
  'simulations',
  'for',
  'various',
  'equipment',
  '-',
  'failure',
  'scenarios',
  ',',
  'and',
  'found',
  'that',
  'the',
  'only',
  'scenario',
  '

### Hievents

In [18]:
import re
from typing import List, Dict, Any
from collections import defaultdict
import json
import pprint

def _tokenize_with_offsets(text: str):
    """
    Split *text* on whitespace, returning both tokens and their
    character offsets (start, end – end exclusive).
    """
    tokens, offsets = [], []
    for m in re.finditer(r"\S+", text):
        tokens.append(m.group(0))
        offsets.append((m.start(), m.end()))
    return tokens, offsets


def prepare_document(raw: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convert one raw document (as in your example) into a dictionary with:
      • id          – str
      • tokens      – List[str]        (original sentences)
      • words       – List[str]        (flat list of all tokens)
      • mentions    – List[int]        (original mention *ids*)
      • spans       – List[List[int]]  (token indices per mention, parallel to *mentions*)
      • relations   – Dict[str, List[List[int]]]
                      (pairs of *mention indices*, **not** mention ids)

    The token indices in *spans* and the pairs in *relations* are all based on
    the flattened *words* list.
    """
    sentences: List[str] = raw["text"]

    # ---- 1. tokenise every sentence, collect global token offsets ----
    words: List[str] = []
    sent_token_meta: List[List[tuple]] = []       # [(tok, start, end, global_idx), ...] per sent

    for sent in sentences:
        toks, offs = _tokenize_with_offsets(sent)
        base_idx = len(words)                     # first global token idx for this sentence
        words.extend(toks)
        sent_token_meta.append(
            [(tok, s, e, base_idx + i) for i, (tok, (s, e)) in enumerate(zip(toks, offs))]
        )

    # ---- 2. build mentions → span (token indices) ----
    mentions: List[int] = []
    spans: List[List[int]] = []
    id_to_pos: Dict[int, int] = {}               # mention id  → position in *mentions*

    for pos, m in enumerate(raw["events"]):
        m_id = m["id"]
        mentions.append(m_id)
        id_to_pos[m_id] = pos

        sent_id = m["sent_id"]
        char_start, char_end = m["offset"]

        # tokens whose char span overlaps the mention’s char span
        tok_indices = [
            glob_idx
            for _tok, t_start, t_end, glob_idx in sent_token_meta[sent_id]
            if not (t_end <= char_start or t_start >= char_end)
        ]
        spans.append(tok_indices)

    # ---- 3. rewrite relations to use mention *positions* ----
    relations: Dict[str, List[List[int]]] = {}
    for rel_type, pairs in raw["relations"].items():
        mapped: List[List[int]] = [
            [id_to_pos[a], id_to_pos[b]]
            for a, b in pairs
            if a in id_to_pos and b in id_to_pos
        ]
        relations[rel_type] = mapped

    # ---- 4. package result ----
    return {
        "id": raw["id"],
        "tokens": words,
        "mentions": mentions,
        "spans": spans,
        "relations": relations,
    }

In [ ]:
# ---------------------------------------------------------------------
# Example usage
# if __name__ == "__main__":
    

example_json = """{
  "id": "article-20575",
  "text": [
    "Two Pakistani air force planes crashed in a residential area in northwestern Pakistan on Thursday, killing all four pilots on board and injuring five people on the ground, police said.",
    "Residents of Nowshera city, where the planes went down, reported that the aircraft collided before they crashed, but the air force is still investigating, said police official Fazil Khan.",
    "The planes took off from an air force academy in nearby Risalpur, said Mohammad Hussain, the Nowshera police chief.",
    "Local TV footage showed the twisted metal wreckage from one of the planes among a group of houses in Nowshera.",
    "The air force has suffered a series of crashes over the past year that it has said were the result of technical problems.",
    ""
  ],
  "events": [
    {"id": 1, "mention": "crashed", "sent_id": 0, "offset": [31, 38]},
    {"id": 2, "mention": "killing", "sent_id": 0, "offset": [99, 106]},
    {"id": 3, "mention": "injuring", "sent_id": 0, "offset": [136, 144]},
    {"id": 4, "mention": "said", "sent_id": 0, "offset": [179, 183]},
    {"id": 5, "mention": "went", "sent_id": 1, "offset": [45, 49]},
    {"id": 6, "mention": "reported", "sent_id": 1, "offset": [56, 64]},
    {"id": 7, "mention": "collided", "sent_id": 1, "offset": [83, 91]},
    {"id": 8, "mention": "crashed", "sent_id": 1, "offset": [104, 111]},
    {"id": 9, "mention": "investigating", "sent_id": 1, "offset": [140, 153]},
    {"id": 10, "mention": "said", "sent_id": 1, "offset": [155, 159]},
    {"id": 11, "mention": "took", "sent_id": 2, "offset": [11, 15]},
    {"id": 12, "mention": "said", "sent_id": 2, "offset": [66, 70]},
    {"id": 13, "mention": "showed", "sent_id": 3, "offset": [17, 23]},
    {"id": 14, "mention": "suffered", "sent_id": 4, "offset": [18, 26]},
    {"id": 15, "mention": "crashes", "sent_id": 4, "offset": [39, 46]},
    {"id": 16, "mention": "said", "sent_id": 4, "offset": [78, 82]}
  ],
  "relations": {
    "SuperSub": [[5, 15], [3, 8], [1, 2], [2, 15], [1, 3], [2, 8]],
    "SubSuper": [[1, 15], [2, 5], [8, 15], [3, 15], [3, 5]],
    "Coref":    [[1, 8], [5, 8], [1, 5]]
  }
}"""

raw_doc = json.loads(example_json)
prepared = prepare_document(raw_doc)
# Pretty-print the result for inspection

pprint.pprint(prepared, width=120)

In [19]:
# ──────────────────────────────────────────────────────────────────────────────
def build_dataset_dict(root_dir: str, exclude: Set[str]) -> DatasetDict:
    """Walk every <language>/<split> directory and build a DatasetDict."""
    split_rows = defaultdict(list)          # split → list[dict]

    for lang_dir in glob.glob(os.path.join(root_dir, "*")):
        if not os.path.isdir(lang_dir):
            continue
        lang = os.path.basename(lang_dir)

        for split in ("train", "test", "dev"):
            pattern = os.path.join(lang_dir, split, "*json")
            for path in glob.glob(pattern):
                if os.path.basename(path) in exclude:
                    print(f"🚫  Skipping excluded file: {path}")
                    continue
                row = _parse_ann_file(path)
                row["lang"]  = lang
                row["split"] = split
                split_rows[split].append(row)

    if not split_rows:
        raise RuntimeError(f"No documents found under {root_dir}")

    # Convert to HuggingFace datasets
    return DatasetDict({sp: Dataset.from_list(rows) for sp, rows in split_rows.items()})

# ──────────────────────────────────────────────────────────────────────────────
def push_to_hub(dd: DatasetDict, repo_id: str, private: bool, token: str | None):
    dd.push_to_hub(repo_id,
                   token=token or os.getenv("HF_TOKEN"),
                   private=private)
    print(f"✅  Dataset pushed to https://huggingface.co/datasets/{repo_id}")

In [38]:
import json

root_dir = 'data/MAVEN-ERE/processed/hievents/'
file_type = '.json'

split_rows = defaultdict(list)
for split in ("train", "test", "dev"):
    path = root_dir + split + file_type
    with open(path, encoding="utf-8") as fh:
        docs = [json.loads(ln) for ln in fh]
    
    for raw_doc in docs:
        prep_doc = prepare_document(raw_doc)

        split_rows[split].append(prep_doc)
    
    if not split_rows:
        raise RuntimeError(f"No documents found under {root_dir}")

In [39]:
ds_hievents = DatasetDict({sp: Dataset.from_list(rows) for sp, rows in split_rows.items()})

In [41]:
for sp in ds_hievents:
    ds_hievents[sp] = ds_hievents[sp].shuffle(seed=42)

push_to_hub(ds_hievents,
            repo_id="Nofing/Hievents-span",
            private=False,
            token=None)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/225 [00:00<?, ?B/s]

✅  Dataset pushed to https://huggingface.co/datasets/Nofing/Hievents-span


In [26]:
split_rows['dev'][0]

{'id': 'article-18043',
 'tokens': ['"Majid Jamali Fashi, the Mossad spy and the person who assassinated Masoud Ali Mohammadi, our nation\'s nuclear scientist, was hanged Tuesday morning (local time)," IRNA said.',
  'Local media reported August 28 that Jamali Fashi was sentenced to death after being "convicted of Moharebeh (waging war against God) for placing a bomb-laden bike and blowing it up in front of martyr Ali Mohammadi\'s home, in collaboration with the Zionist regime and Mossad".',
  'Jamali Fashi stood trial as the main suspect in the killing of Ali Mohammadi, a particle physics professor at Tehran University who was killed in a bomb attack outside his home in January 2010.',
  "Jamali Fashi also faced charges of cooperating with Israel's spy agency Mossad and of receiving $US120,000 for passing on intelligence to its agents.",
  'The Islamic republic has blamed the Jewish state and the United States for the killing of four of its scientists and nuclear experts since 2010.',

In [ ]:
exclude_set = []

ds_dict = build_dataset_dict("data/MECI/meci-v0.1-public", exclude_set)

# Optional shuffle for reproducible training

for sp in ds_dict:
    ds_dict[sp] = ds_dict[sp].shuffle(seed=42)

push_to_hub(ds_dict,
            repo_id="Nofing/MECI-v0.1-public-span",
            private=False,
            token=None)